# DAMU Evaluation Suite (Colab)

Drive-synced Colab notebook for evaluating models / test-time methods on **Dynamic Audio Motion Understanding (DAMU)**.

## Experiment design (what this notebook runs)

| ID | Experiment | Goal |
| --- | --- | --- |
| **E0** | Dataset inventory + label audit | Confirm Drive corpora, sample counts, label coverage |
| **E1** | Difficulty / skill proxy audit | Mirror MMAU-style analysis: are DAMU items harder / more motion-reasoning? |
| **E2** | Acoustic feature baselines | Cheap, offline baselines (majority / spectrogram features) |
| **E3** | SE-ResNet B1 speed baseline | Paper baseline: speed RMSE/MAE on Real / Synth / Mixed if weights exist |
| **E5** | Test-time method harness | Score external predictions (zero-shot / CoT / self-consistency / your method) |

## Setup flow

1. Mount Drive + edit paths
2. Clone / pull `DAMU_neurips26` with GitHub token
3. Install deps
4. Inventory data
5. Run selected experiments

> Tip: set `RUN_*` flags in the config cell to only run what you need.




## 0 — Mount Drive + path config


In [ ]:
from google.colab import drive
from getpass import getpass
import os
from pathlib import Path

drive.mount("/content/drive")

# ---- edit these ----
# Shared Drive root that holds Datasets/ (or a local copy on Drive)
DRIVE_ROOT = "/content/drive/Shareddrives/Spectral Transformers - Doppler/DopplerLab"
# Preferred: one parent folder that contains all dataset subfolders
DATASETS_DIR = f"{DRIVE_ROOT}/datasets/DAMU"
# Fallback: repo Datasets/ copy on Drive
DATASETS_DIR_FALLBACK = f"{DRIVE_ROOT}/DAMU_neurips26/Datasets"

# If you uploaded SEPARATE Drive folders (recommended), set these explicitly.
# Leave as None to auto-resolve under DATASETS_DIR.
REAL_DATA_DIR = None          # e.g. f"{DRIVE_ROOT}/datasets/DAMU_RealData"
SIM_DATA_DIR = None           # e.g. f"{DRIVE_ROOT}/datasets/DAMU_SimulatedData"
BENCH_DATA_DIR = None         # e.g. f"{DRIVE_ROOT}/datasets/DAMU_Benchmarks"

# Git
GIT_REPO = "https://github.com/seetharamkkv/DAMU_neurips26.git"
GIT_BRANCH = "master"
REPO_DIR = "/content/DAMU_neurips26"

# Outputs (created on Drive so runs persist)
RUN_TAG = "damu_eval_v1"
RUN_ROOT = f"{DRIVE_ROOT}/DAMU_evals/{RUN_TAG}"

# Experiment switches
RUN_E0_INVENTORY = True
RUN_E1_DIFFICULTY = True
RUN_E2_FEATURE_BASELINES = True
RUN_E3_SERESNET_B1 = True
RUN_E5_TTA_HARNESS = True

# Subsampling for quick Colab runs (None = all samples)
MAX_SAMPLES_PER_TASK = 64
SEED = 42

# Optional: path to external prediction CSVs for E5
# Expected columns: sample_id, task_id, prediction  (+ optional method)
EXTERNAL_PREDS_DIR = f"{RUN_ROOT}/external_preds"

TASKS = [
    "B1_SpeedEstimation",
    "B2_DirectionIdentification",
    "B3_DistanceEstimation",
    "B4_TrajectoryLocalization",
    "B5_Time-to-EventPrediction",
    "B6_MotionStateSegmentation",
    "B7_AcceleratedMotionAnalysis",
    "B8_Multi-ObjectResolution",
    "B9_InteractionModeling",
    "B10_SourceIdentityRecognition",
]

os.makedirs(RUN_ROOT, exist_ok=True)
os.makedirs(EXTERNAL_PREDS_DIR, exist_ok=True)

print("DRIVE_ROOT:   ", DRIVE_ROOT)
print("DATASETS_DIR: ", DATASETS_DIR)
print("RUN_ROOT:     ", RUN_ROOT)
print("Git:          ", GIT_REPO, "@", GIT_BRANCH)
print("Tasks:", len(TASKS), "| max/task:", MAX_SAMPLES_PER_TASK)
print("Flags: E0=%s E1=%s E2=%s E3=%s E5=%s" % (
    RUN_E0_INVENTORY, RUN_E1_DIFFICULTY, RUN_E2_FEATURE_BASELINES,
    RUN_E3_SERESNET_B1, RUN_E5_TTA_HARNESS,
))





## 1 — Clone / pull GitHub repo (token) + install deps


In [ ]:
import shutil
import subprocess
from tqdm.auto import tqdm

def run_cmd(cmd, cwd=None, desc=None):
    if desc:
        print(f"\n>>> {desc}")
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd, cwd=cwd)

# Optional token for private repo (leave blank if public / already authed)
GH_TOKEN = getpass("GitHub token (blank if public): ").strip()
clone_url = GIT_REPO
if GH_TOKEN:
    clone_url = GIT_REPO.replace("https://", f"https://{GH_TOKEN}@")

pbar = tqdm(total=3, desc="Repo setup", unit="step")

if Path(REPO_DIR).is_dir():
    print("Repo exists:", REPO_DIR)
    run_cmd(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], desc="Set remote URL")
    run_cmd(["git", "-C", REPO_DIR, "fetch", "origin", GIT_BRANCH], desc="Fetch branch")
    run_cmd(["git", "-C", REPO_DIR, "checkout", GIT_BRANCH], desc="Checkout")
    run_cmd(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", GIT_BRANCH], desc="Pull ff-only")
else:
    run_cmd([
        "git", "clone", "--branch", GIT_BRANCH, "--single-branch", clone_url, REPO_DIR
    ], desc="Clone repo")
pbar.update(2)

os.chdir(REPO_DIR)
head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
print(f"HEAD: {head} | Branch: {branch}")

req = Path(REPO_DIR) / "requirements.txt"
assert req.is_file(), f"Missing requirements: {req}"
print("\n>>> Installing Python deps (quiet)")
%pip install -q -r requirements.txt
%pip install -q tqdm rich soundfile librosa scikit-learn matplotlib seaborn
pbar.update(1)
pbar.close()

SERESNET_DIR = Path(REPO_DIR) / "Vehicle-Speed-from-Audio-SE-ResNet"
DOPPLERSIM_DIR = Path(REPO_DIR) / "DopplerSim"
assert SERESNET_DIR.is_dir(), SERESNET_DIR
assert DOPPLERSIM_DIR.is_dir(), DOPPLERSIM_DIR
print("Deps installed. Repo ready.")


## 2 — Resolve datasets + helpers

Looks for `Benchmarks/` under Drive (primary) or the cloned repo `Datasets/` (fallback).


In [ ]:
import json
import warnings
from collections import Counter
from typing import Optional

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

TASK_META = {
    "B1_SpeedEstimation": {
        "family": "kinematic", "type": "regression", "target": "speed_mps",
        "metrics": ["mae", "rmse"],
        "reasoning": ["Temporal Reasoning", "Fine-Grained Acoustic Discrimination"],
        "perception": ["Acoustic & Environmental Source Knowledge"],
    },
    "B2_DirectionIdentification": {
        "family": "kinematic", "type": "classification", "target": "direction_label",
        "metrics": ["accuracy", "macro_f1"],
        "reasoning": ["Temporal Reasoning", "Causal Reasoning"],
        "perception": ["Acoustic & Environmental Source Knowledge"],
    },
    "B3_DistanceEstimation": {
        "family": "geometric", "type": "regression", "target": "cpa_distance_m",
        "metrics": ["mae", "rmse"],
        "reasoning": ["Information Integration", "Spatial Reasoning"],
        "perception": ["Acoustic & Environmental Source Knowledge"],
    },
    "B4_TrajectoryLocalization": {
        "family": "geometric", "type": "classification", "target": "trajectory_type",
        "metrics": ["accuracy", "macro_f1"],
        "reasoning": ["Temporal Reasoning", "Information Integration"],
        "perception": ["Acoustic & Environmental Source Knowledge"],
    },
    "B5_Time-to-EventPrediction": {
        "family": "temporal", "type": "regression", "target": "cpa_time_sec",
        "metrics": ["mae", "rmse"],
        "reasoning": ["Temporal Reasoning", "Causal Reasoning"],
        "perception": ["Acoustic & Environmental Source Knowledge"],
    },
    "B6_MotionStateSegmentation": {
        "family": "temporal", "type": "sequence", "target": "direction_label",
        "metrics": ["accuracy"],
        "reasoning": ["Temporal Reasoning", "Information Integration"],
        "perception": ["Acoustic & Environmental Source Knowledge"],
    },
    "B7_AcceleratedMotionAnalysis": {
        "family": "kinematic", "type": "regression", "target": "acceleration_mps2",
        "metrics": ["mae", "rmse"],
        "reasoning": ["Temporal Reasoning", "Fine-Grained Acoustic Discrimination"],
        "perception": ["Acoustic & Environmental Source Knowledge"],
    },
    "B8_Multi-ObjectResolution": {
        "family": "scene", "type": "regression", "target": "num_sources",
        "metrics": ["mae", "accuracy"],
        "reasoning": ["Multi-Source Separation", "Information Integration", "Counting"],
        "perception": ["Multi-Source Separation"],
    },
    "B9_InteractionModeling": {
        "family": "scene", "type": "classification", "target": "is_crossing",
        "metrics": ["accuracy", "macro_f1"],
        "reasoning": ["Causal Reasoning", "Temporal Reasoning", "Ambiguity Resolution"],
        "perception": ["Multi-Source Separation"],
    },
    "B10_SourceIdentityRecognition": {
        "family": "scene", "type": "classification", "target": "vehicle_class",
        "metrics": ["accuracy", "macro_f1"],
        "reasoning": ["Information Integration", "Elimination of Alternatives"],
        "perception": ["Acoustic & Environmental Source Knowledge"],
    },
}

def resolve_datasets_dir():
    # If user pointed BENCH_DATA_DIR directly, synthesize a virtual root.
    if BENCH_DATA_DIR and Path(BENCH_DATA_DIR).is_dir():
        return Path(BENCH_DATA_DIR).parent
    candidates = [Path(DATASETS_DIR), Path(DATASETS_DIR_FALLBACK), Path(REPO_DIR) / "Datasets"]
    for c in candidates:
        if (c / "Benchmarks").is_dir() or (c / "RealData").is_dir():
            return c
    raise FileNotFoundError(
        "Could not find Datasets. Edit DATASETS_DIR or the separate *_DIR paths. Tried:\n  - "
        + "\n  - ".join(str(c) for c in candidates)
    )

DATA_ROOT = resolve_datasets_dir()
print("DATA_ROOT:", DATA_ROOT)

def _pick_dir(*cands):
    for c in cands:
        if c and Path(c).is_dir():
            return Path(c)
    return None

# Allow separately uploaded Drive folders
if BENCH_DATA_DIR:
    BENCH_DIR = Path(BENCH_DATA_DIR)
else:
    BENCH_DIR = DATA_ROOT / "Benchmarks"

REAL_DIR = _pick_dir(REAL_DATA_DIR, DATA_ROOT / "RealData")
SIM_DIR = _pick_dir(SIM_DATA_DIR, DATA_ROOT / "SimulatedData")
print("Real:", REAL_DIR)
print("Sim:", SIM_DIR)
print("Bench:", BENCH_DIR, "exists=", BENCH_DIR.exists() if BENCH_DIR else None)


def task_csv(task: str) -> Path:
    root = BENCH_DIR
    aliases = [task]
    if task == "B8_Multi-ObjectResolution":
        aliases += ["B8_MultiObjectResolution", "B8_Multi-ObjectResolution"]
    for name in aliases:
        p = root / name / "dataset.csv"
        if p.is_file():
            return p
    return root / task / "dataset.csv"


def resolve_sample_wav(sample_dir: Path, filename: str) -> Path:
    """Benchmarks store wavs under Essential/ or Common/, not sample root."""
    candidates = [
        sample_dir / "Essential" / filename,
        sample_dir / "Common" / filename,
        sample_dir / filename,
    ]
    for c in candidates:
        if c.is_file():
            return c
    hits = list(sample_dir.rglob(filename)) if sample_dir.is_dir() else []
    if hits:
        hits = sorted(hits, key=lambda x: (0 if "Essential" in str(x) else 1, str(x)))
        return hits[0]
    return candidates[0]


def load_task_df(task: str, max_n: Optional[int] = None) -> pd.DataFrame:
    csv_path = task_csv(task)
    if not csv_path.is_file():
        raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path)
    df["task_id"] = task
    audio_root = csv_path.parent / "audio_clips"
    paths = []
    for _, row in df.iterrows():
        sample_dir = audio_root / row["sample_id"]
        wav = resolve_sample_wav(sample_dir, row["filename"])
        paths.append(str(wav))
    df["audio_path"] = paths
    if max_n is not None and len(df) > max_n:
        df = df.sample(n=max_n, random_state=SEED).reset_index(drop=True)
    return df


def status_banner(title: str):
    bar = "=" * 72
    print(f"\n{bar}\n{title}\n{bar}")


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    print("Saved:", path)

print("Helpers ready. TASK_META covers", len(TASK_META), "tasks.")







## E0 — Dataset inventory + label audit

Counts samples, checks audio presence, and summarizes label distributions per task.


In [ ]:
if not RUN_E0_INVENTORY:
    print("Skipped E0 (RUN_E0_INVENTORY=False)")
else:
    status_banner("E0 — Dataset inventory")
    rows = []
    label = "clean"
    root = BENCH_DIR
    if not root.is_dir():
        raise FileNotFoundError(f"Missing Benchmarks root: {root}")
    for task in tqdm(TASKS, desc=f"Inventory ({label})"):
        csv_path = task_csv(task)
        if not csv_path.is_file():
            rows.append({"split": label, "task": task, "n": 0, "audio_ok": 0, "status": "MISSING_CSV"})
            continue
        df = pd.read_csv(csv_path)
        audio_root = csv_path.parent / "audio_clips"
        ok = 0
        for _, r in tqdm(df.iterrows(), total=len(df), desc=f"{label}/{task}", leave=False):
            sample_dir = audio_root / r["sample_id"]
            wav = resolve_sample_wav(sample_dir, r["filename"])
            if Path(wav).is_file():
                ok += 1
        rows.append({
            "split": label,
            "task": task,
            "n": len(df),
            "audio_ok": ok,
            "status": "OK" if ok == len(df) else "PARTIAL",
            "target": TASK_META[task]["target"],
            "type": TASK_META[task]["type"],
        })
    inv = pd.DataFrame(rows)
    display(inv)
    inv_path = Path(RUN_ROOT) / "E0_inventory.csv"
    inv.to_csv(inv_path, index=False)
    print("Saved:", inv_path)

    status_banner("E0 — Label distributions (clean, subsampled)")
    label_report = {}
    for task in tqdm(TASKS, desc="Label audit"):
        try:
            df = load_task_df(task, max_n=MAX_SAMPLES_PER_TASK)
        except FileNotFoundError:
            continue
        tgt = TASK_META[task]["target"]
        if tgt not in df.columns:
            label_report[task] = {"error": f"missing column {tgt}"}
            continue
        s = df[tgt]
        if TASK_META[task]["type"] == "classification" or s.dtype == object or str(s.dtype).startswith("bool"):
            vc = s.astype(str).value_counts().to_dict()
            label_report[task] = {"n": len(df), "value_counts": vc}
        else:
            label_report[task] = {
                "n": len(df),
                "min": float(np.nanmin(s)),
                "max": float(np.nanmax(s)),
                "mean": float(np.nanmean(s)),
                "std": float(np.nanstd(s)),
            }
    save_json(label_report, Path(RUN_ROOT) / "E0_label_report.json")
    for k, v in label_report.items():
        print(f"{k}: {v}")




## E1 — Difficulty / skill proxy audit

Mirrors the collaborator's MMAU analysis, but for **DAMU**.

- **Skill tags** come from the task taxonomy (motion / geometric / multi-source), not speaker-gender perception.
- **Difficulty proxy** is acoustic+label based (no API required): speed, CPA distance, multi-source count, acceleration, trajectory complexity → 1–10 scale.

Optional later: swap in model-judged difficulty with an omni API the same way they did.


In [ ]:
import matplotlib.pyplot as plt

def difficulty_proxy(row, task: str) -> float:
    # Heuristic 1-10 difficulty for DAMU items (physics / scene complexity).
    score = 3.0
    speed = float(row.get("speed_mps", 0) or 0)
    score += np.clip((speed - 10) / 40 * 2.5, 0, 2.5)
    dist = float(row.get("cpa_distance_m", 50) or 50)
    score += np.clip(abs(np.log10(max(dist, 1.0) / 25.0)) * 1.5, 0, 2.0)
    acc = abs(float(row.get("acceleration_mps2", 0) or 0))
    score += np.clip(acc / 3.0, 0, 1.5)
    traj = str(row.get("trajectory_type", "straight"))
    score += {"straight": 0.0, "parabola": 0.6, "bezier": 1.0, "busy_road": 1.8}.get(traj, 0.5)
    nsrc = int(row.get("num_sources", 1) or 1)
    score += max(0, nsrc - 1) * 0.7
    if bool(row.get("is_crossing", False)):
        score += 1.2
    family_bump = {"kinematic": 0.0, "geometric": 0.4, "temporal": 0.6, "scene": 1.5}
    score += family_bump.get(TASK_META[task]["family"], 0.0)
    return float(np.clip(score, 1.0, 10.0))


if not RUN_E1_DIFFICULTY:
    print("Skipped E1")
else:
    status_banner("E1 — Difficulty / skill proxy audit")
    records = []
    skill_rows = []
    for task in tqdm(TASKS, desc="E1 tasks"):
        try:
            df = load_task_df(task, max_n=MAX_SAMPLES_PER_TASK)
        except FileNotFoundError as e:
            print("skip", task, e)
            continue
        for _, row in df.iterrows():
            d = difficulty_proxy(row, task)
            records.append({
                "task": task,
                "family": TASK_META[task]["family"],
                "sample_id": row["sample_id"],
                "difficulty": d,
                "num_sources": int(row.get("num_sources", 1) or 1),
            })
        for sk in TASK_META[task]["reasoning"]:
            skill_rows.append({"task": task, "skill_type": "reasoning", "skill": sk, "pct": 100.0})
        for sk in TASK_META[task]["perception"]:
            skill_rows.append({"task": task, "skill_type": "perception", "skill": sk, "pct": 100.0})

    diff_df = pd.DataFrame(records)
    skill_df = pd.DataFrame(skill_rows)
    diff_df.to_csv(Path(RUN_ROOT) / "E1_difficulty_per_sample.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    order = ["kinematic", "geometric", "temporal", "scene"]
    data = [diff_df.loc[diff_df.family == f, "difficulty"].values for f in order if (diff_df.family == f).any()]
    labels = [f for f in order if (diff_df.family == f).any()]
    axes[0].boxplot(data, labels=labels, patch_artist=True)
    axes[0].set_title("DAMU difficulty proxy by task family")
    axes[0].set_ylabel("Difficulty (1–10)")
    axes[0].set_ylim(1, 10)
    axes[0].axhline(5.5, ls="--", c="gray", alpha=0.6, label="MMAU-ish mid band")
    axes[0].legend(fontsize=8)

    reason = skill_df[skill_df.skill_type == "reasoning"]
    cov = reason.groupby("skill")["task"].nunique().sort_values(ascending=True)
    cov_pct = 100.0 * cov / max(diff_df["task"].nunique(), 1)
    axes[1].barh(cov_pct.index, cov_pct.values, color="#2a9d8f")
    axes[1].set_xlabel("% of DAMU tasks requiring skill")
    axes[1].set_title("Reasoning skill coverage (taxonomy)")
    plt.tight_layout()
    fig_path = Path(RUN_ROOT) / "E1_difficulty_skills.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)

    summary = diff_df.groupby("family")["difficulty"].agg(["count", "mean", "median", "std"]).round(3)
    display(summary)
    print("Overall mean difficulty:", round(diff_df.difficulty.mean(), 3))
    print("Share >= 6.0:", round((diff_df.difficulty >= 6).mean() * 100, 1), "%")



## E2 — Acoustic feature baselines (offline)

Runnable without any API. Establishes floors for each task:

1. **Majority / mean** chance baseline
2. **Spectrogram summary features + Ridge / LogisticRegression**

These are *not* the paper's main model — they quantify how solvable DAMU is from shallow cues.


In [ ]:
import librosa
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def load_wav_mono(path, sr=16000, max_sec=10.0):
    y, file_sr = sf.read(path, always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if file_sr != sr:
        y = librosa.resample(y.astype(np.float32), orig_sr=file_sr, target_sr=sr)
    n = int(sr * max_sec)
    if len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    else:
        y = y[:n]
    return y.astype(np.float32), sr


def wav_features(path):
    y, sr = load_wav_mono(path)
    S = np.abs(librosa.stft(y, n_fft=512, hop_length=256))
    bands = np.array_split(S, 8, axis=0)
    feats = []
    for b in bands:
        feats += [b.mean(), b.std()]
    cent = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    feats += [cent.mean(), cent.std(), cent[-1] - cent[0], np.percentile(cent, 90) - np.percentile(cent, 10)]
    rms = librosa.feature.rms(y=y)[0]
    feats += [rms.mean(), rms.std(), rms.argmax() / max(len(rms) - 1, 1)]
    return np.asarray(feats, dtype=np.float32)


def evaluate_task_baseline(task: str):
    meta = TASK_META[task]
    df = load_task_df(task, max_n=MAX_SAMPLES_PER_TASK)
    tgt = meta["target"]
    y = df[tgt].values
    is_clf = meta["type"] in ("classification", "sequence") or str(df[tgt].dtype) == "bool" or df[tgt].dtype == object

    X, keep = [], []
    for i, p in enumerate(tqdm(df["audio_path"].tolist(), desc=f"feats {task}", leave=False)):
        if not Path(p).is_file():
            continue
        try:
            X.append(wav_features(p))
            keep.append(i)
        except Exception as e:
            print("  feature fail", Path(p).name, e)
    if len(keep) < 4:
        return {"task": task, "status": "TOO_FEW", "n": len(keep)}

    X = np.stack(X)
    y = y[keep]
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, test_size=0.3, random_state=SEED)
    out = {"task": task, "n": int(len(y)), "status": "OK"}

    if is_clf:
        y = np.asarray([str(v) for v in y])
        maj = Counter(y[tr]).most_common(1)[0][0]
        maj_pred = np.array([maj] * len(te))
        out["majority_acc"] = float(accuracy_score(y[te], maj_pred))
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        try:
            clf.fit(X[tr], y[tr])
            pred = clf.predict(X[te])
            out["feat_acc"] = float(accuracy_score(y[te], pred))
            out["feat_macro_f1"] = float(f1_score(y[te], pred, average="macro"))
        except Exception as e:
            out["feat_error"] = str(e)
    else:
        y = y.astype(np.float64)
        mean_pred = np.full(len(te), y[tr].mean())
        out["mean_mae"] = float(mean_absolute_error(y[te], mean_pred))
        out["mean_rmse"] = float(np.sqrt(mean_squared_error(y[te], mean_pred)))
        reg = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
        reg.fit(X[tr], y[tr])
        pred = reg.predict(X[te])
        out["feat_mae"] = float(mean_absolute_error(y[te], pred))
        out["feat_rmse"] = float(np.sqrt(mean_squared_error(y[te], pred)))
    return out


if not RUN_E2_FEATURE_BASELINES:
    print("Skipped E2")
else:
    status_banner("E2 — Acoustic feature baselines")
    e2_rows = []
    for task in tqdm(TASKS, desc="E2 tasks"):
        try:
            e2_rows.append(evaluate_task_baseline(task))
        except Exception as e:
            e2_rows.append({"task": task, "status": f"ERROR: {e}"})
    e2 = pd.DataFrame(e2_rows)
    display(e2)
    e2_path = Path(RUN_ROOT) / "E2_feature_baselines.csv"
    e2.to_csv(e2_path, index=False)
    print("Saved:", e2_path)




## E3 — SE-ResNet B1 speed baseline (paper)

Runs the existing `batch_inference.py` 3×3 grid **if** checkpoints + Real/Simulated data are present.
Otherwise prints what is missing and skips cleanly.


In [ ]:
if not RUN_E3_SERESNET_B1:
    print("Skipped E3")
else:
    status_banner("E3 — SE-ResNet B1 speed baseline")
    ser = SERESNET_DIR
    ckpt = ser / "checkpoints"
    real = REAL_DIR or (DATA_ROOT / "RealData")
    sim = SIM_DIR or (DATA_ROOT / "SimulatedData")

    print("SE-ResNet dir:", ser)
    print("checkpoints:  ", ckpt.is_dir(), "|", list(ckpt.glob("*"))[:8] if ckpt.is_dir() else None)
    print("RealData:     ", real.is_dir())
    print("SimulatedData:", sim.is_dir())

    batch_script = ser / "batch_inference.py"
    has_weights = ckpt.is_dir() and (any(ckpt.rglob("*.h5")) or any(ckpt.rglob("*.weights*")))
    if not batch_script.is_file():
        print("Missing batch_inference.py — skip")
    elif not has_weights:
        print("No SE-ResNet weights found under checkpoints/ — skip E3")
        print("Place weights in:", ckpt)
    else:
        link = ser.parent / "Datasets"
        if not link.exists():
            try:
                os.symlink(DATA_ROOT, link)
                print("Linked Datasets ->", DATA_ROOT)
            except Exception as e:
                print("Could not symlink Datasets:", e)

        out_log = Path(RUN_ROOT) / "E3_seresnet_batch_inference.log"
        print("Running batch_inference.py ...")
        with open(out_log, "w", encoding="utf-8") as logf:
            proc = subprocess.Popen(
                ["python", "batch_inference.py"],
                cwd=str(ser),
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            assert proc.stdout is not None
            for line in tqdm(proc.stdout, desc="SE-ResNet batch_inference"):
                print(line, end="")
                logf.write(line)
            rc = proc.wait()
        print("exit:", rc, "| log:", out_log)



## E5 — Test-time method harness

Collaboration-facing eval entrypoint.

Drop prediction CSVs into `EXTERNAL_PREDS_DIR` with columns:

```text
sample_id,task_id,prediction,method
```

Example methods: `zeroshot`, `cot`, `self_consistency`, `your_tta_method`.

If no external preds exist yet, writes a **template CSV** + mock method so the pipeline is verified end-to-end.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score

def score_predictions(pred_df: pd.DataFrame, gt_by_task: dict) -> pd.DataFrame:
    rows = []
    for (method, task), g in pred_df.groupby(["method", "task_id"]):
        if task not in gt_by_task:
            rows.append({"method": method, "task": task, "status": "NO_GT"})
            continue
        gt = gt_by_task[task]
        meta = TASK_META[task]
        tgt = meta["target"]
        merged = g.merge(gt[["sample_id", tgt]], on="sample_id", how="inner")
        if merged.empty:
            rows.append({"method": method, "task": task, "status": "NO_OVERLAP", "n": 0})
            continue
        y_true = merged[tgt]
        y_pred = merged["prediction"]
        row = {"method": method, "task": task, "n": len(merged), "status": "OK", "type": meta["type"]}
        if meta["type"] in ("classification", "sequence") or y_true.dtype == object or str(y_true.dtype) == "bool":
            yt = y_true.astype(str)
            yp = y_pred.astype(str)
            row["accuracy"] = float(accuracy_score(yt, yp))
            row["macro_f1"] = float(f1_score(yt, yp, average="macro"))
        else:
            yt = y_true.astype(float)
            yp = pd.to_numeric(y_pred, errors="coerce")
            mask = yp.notna()
            yt, yp = yt[mask], yp[mask]
            row["mae"] = float(mean_absolute_error(yt, yp))
            row["rmse"] = float(np.sqrt(mean_squared_error(yt, yp)))
        rows.append(row)
    return pd.DataFrame(rows)


if not RUN_E5_TTA_HARNESS:
    print("Skipped E5")
else:
    status_banner("E5 — Test-time method harness")
    pred_dir = Path(EXTERNAL_PREDS_DIR)
    pred_files = sorted(pred_dir.glob("*.csv"))

    gt_by_task = {}
    for task in tqdm(TASKS, desc="Load GT"):
        try:
            gt_by_task[task] = load_task_df(task, max_n=None)
        except FileNotFoundError:
            pass

    if not pred_files:
        print("No external preds found — writing template + mock method")
        rng = np.random.RandomState(SEED)
        mock_rows = []
        for task, gt in gt_by_task.items():
            tgt = TASK_META[task]["target"]
            sub = gt.head(min(16, len(gt)))
            for _, r in sub.iterrows():
                if TASK_META[task]["type"] in ("classification", "sequence") or gt[tgt].dtype == object:
                    pred = str(r[tgt])
                else:
                    pred = float(r[tgt]) * (1.0 + rng.randn() * 0.1)
                mock_rows.append({
                    "sample_id": r["sample_id"],
                    "task_id": task,
                    "prediction": pred,
                    "method": "mock_noisy_gt",
                })
        template = pd.DataFrame(mock_rows)
        tmpl_path = pred_dir / "template_mock_noisy_gt.csv"
        template.to_csv(tmpl_path, index=False)
        print("Wrote template:", tmpl_path)
        print("Columns required: sample_id, task_id, prediction, method")
        pred_files = [tmpl_path]

    all_preds = []
    for f in tqdm(pred_files, desc="Load pred CSVs"):
        df = pd.read_csv(f)
        need = {"sample_id", "task_id", "prediction"}
        missing = need - set(df.columns)
        if missing:
            print("Skip", f.name, "missing", missing)
            continue
        if "method" not in df.columns:
            df["method"] = f.stem
        all_preds.append(df)
        print(f"  {f.name}: {len(df)} rows | methods={sorted(df['method'].astype(str).unique())}")

    pred_df = pd.concat(all_preds, ignore_index=True)
    scores = score_predictions(pred_df, gt_by_task)
    display(scores)
    out = Path(RUN_ROOT) / "E5_tta_method_scores.csv"
    scores.to_csv(out, index=False)
    print("Saved:", out)

    if not scores.empty and "status" in scores.columns:
        ok = scores[scores.status == "OK"].copy()
        if not ok.empty:
            def _score(r):
                if "accuracy" in r and pd.notna(r.get("accuracy")):
                    return r["accuracy"]
                if pd.notna(r.get("mae")):
                    return -r["mae"]
                return np.nan
            ok["score"] = ok.apply(_score, axis=1)
            piv = ok.pivot_table(index="method", columns="task", values="score", aggfunc="mean")
            print("\nLeaderboard (acc or -MAE):")
            display(piv.round(3))




## 3 — Write run summary

Aggregates flags, paths, and artifact list for sharing with collaborators.


In [ ]:
status_banner("Run summary")
artifacts = sorted(Path(RUN_ROOT).glob("*"))
summary = {
    "run_tag": RUN_TAG,
    "run_root": RUN_ROOT,
    "data_root": str(DATA_ROOT),
    "repo": GIT_REPO,
    "branch": GIT_BRANCH,
    "head": subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip(),
    "max_samples_per_task": MAX_SAMPLES_PER_TASK,
    "flags": {
        "E0": RUN_E0_INVENTORY,
        "E1": RUN_E1_DIFFICULTY,
        "E2": RUN_E2_FEATURE_BASELINES,
        "E3": RUN_E3_SERESNET_B1,
        "E5": RUN_E5_TTA_HARNESS,
    },
    "artifacts": [str(p.name) for p in artifacts if p.is_file()],
}
save_json(summary, Path(RUN_ROOT) / "run_summary.json")
print("Artifacts in", RUN_ROOT)
for a in summary["artifacts"]:
    print(" -", a)
print("\nDone. Share RUN_ROOT with collaborators for E5 prediction drops.")




## How collaborators should plug in their test-time method

1. Run their method on DAMU audio (any model / prompting / TTA).
2. Export a CSV:

```csv
sample_id,task_id,prediction,method
sample_0000001,B1_SpeedEstimation,21.4,cot
sample_0000001,B2_DirectionIdentification,approaching,cot
```

3. Upload to `EXTERNAL_PREDS_DIR` on Drive.
4. Re-run **E5** only (`RUN_E5_TTA_HARNESS=True`, others False).

### Suggested comparison matrix

| Method | B1 RMSE | B2 Acc | B3 MAE | B8 count Acc | B9 Acc | Mean rank |
| --- | --- | --- | --- | --- | --- | --- |
| Zero-shot omni | | | | | | |
| CoT / multi-step | | | | | | |
| Self-consistency | | | | | | |
| Their TTA method | | | | | | |
| Feature baseline (E2) | | | | | | |
| SE-ResNet (B1 only) | | | | | | |
